# Style comparison: reconstruction accuracy

Compare four styles on the same 90 titles, 4×4 PG/TG matrix and two prompt × two image
seeds. Set the four completed jobs, restart the kernel and **Run All**.
Figures appear inline and are saved as PNG/PDF, with CSV tables and provenance.

Primary accuracy uses the prompt check, title-aware image verification and Strict
Exact Match, with blind-strict and normalized alternatives. All planned observations
count, including rejected, failed and missing outcomes as zero. Overall results weight
cells equally, with pointwise 95% intervals from 10,000 paired whole-title bootstrap
samples within domains (seed 20260829).

For metadata-only copies, `ANALYSIS_METADATA_ONLY=1` skips only local image-file checks,
not verification or planned observations. Images remain necessary for visual inspection
and full archiving.


In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from semantic_roundtrip.analysis.plotting import (
    heatmap,
    paired_style_analysis,
    save_figure,
    style_accuracy_panel,
)
from semantic_roundtrip.analysis.reporting import (
    DOMAINS,
    QG,
    SENSITIVITY_METRICS,
    export_tables,
    load_style_jobs,
    prompt_verifications,
    style_accuracy,
    technical_tables,
    write_manifest,
)

# Silence only the known pandas deprecation; data/errors are not suppressed.
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*",
)

# Run from the repository root or its notebooks/ directory.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sns.set_theme(
    style="whitegrid",
    context="notebook",
    rc={"pdf.fonttype": 42, "ps.fonttype": 42, "axes.unicode_minus": False},
)
from semantic_roundtrip.analysis.style_report import (
    STYLE_ANALYSIS_METHODS,
    STYLE_BOOTSTRAP,
    STYLE_PRIMARY_DEFINITION,
    STYLE_PRIMARY_METRIC,
    export_style_summary,
    qwen_pg_style_followup,
    style_centrality,
    validate_style_grid,
)


In [ ]:
# Use a short Windows path for compact copies, e.g. C:/sr/style/<job-directory>.
METADATA_ONLY = os.getenv("ANALYSIS_METADATA_ONLY", "0")
if METADATA_ONLY not in {"0", "1"}:
    raise ValueError("ANALYSIS_METADATA_ONLY must be 0 or 1.")
REQUIRE_IMAGE_FILES = METADATA_ONLY == "0"
print("Local image-file checks:", "required" if REQUIRE_IMAGE_FILES else "omitted (metadata-only copy)")
FREE_JOB = os.getenv("FREE_JOB", "/absolute/path/to/final-direct-core")
SKETCH_JOB = os.getenv("SKETCH_JOB", "/absolute/path/to/final-direct-sketch")
COMIC_JOB = os.getenv("COMIC_JOB", "/absolute/path/to/final-direct-comic")
PHOTOREALISTIC_JOB = os.getenv(
    "PHOTOREALISTIC_JOB", "/absolute/path/to/final-direct-photorealistic"
)
STYLES = {
    "Unrestricted": FREE_JOB,
    "Sketch": SKETCH_JOB,
    "Comic": COMIC_JOB,
    "Photorealistic": PHOTOREALISTIC_JOB,
}
OUTPUT_DIR = (
    Path(os.getenv("OUTPUT_DIR", ROOT / "notebooks/results/style_decision"))
    .expanduser()
    .resolve()
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
jobs, observations, titles = load_style_jobs(STYLES, require_image_files=REQUIRE_IMAGE_FILES)
title_scores = pd.concat(titles.values(), ignore_index=True)
validate_style_grid(title_scores)
accuracy = style_accuracy(title_scores)
centrality = style_centrality(title_scores)

## Overall and by domain

Points show accuracy and bars show pointwise 95% intervals for all four scoring variants.
Unrestricted generation is the reference, not a winner selected by this report.


In [ ]:
domains = ["all", *DOMAINS]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), sharey=True, layout="constrained")
for domain, ax in zip(domains, axes):
    style_accuracy_panel(
        ax, accuracy[accuracy.model_pair.eq("all") & accuracy.domain.eq(domain)], STYLES
    )
    ax.set_title("Overall" if domain == "all" else domain.title())
axes[0].set_ylabel("End-to-end accuracy (%), pointwise 95% CI")
axes[-1].legend(loc="upper right", fontsize=8)
save_figure(
    fig,
    OUTPUT_DIR / "style_accuracy_overall_domains",
    "Style comparison: End-to-end accuracy by domain",
)

## Complete PG/TG matrices

These heatmaps show whether the pooled style result hides PG/TG-specific patterns.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4.4), layout="constrained")
for (style, scores), ax in zip(titles.items(), axes):
    image = heatmap(ax, scores, QG, style)
fig.colorbar(image, ax=list(axes), label="End-to-end Strict Exact Match (%)")
save_figure(
    fig,
    OUTPUT_DIR / "style_accuracy_pg_bi_matrices",
    "Style comparison: complete direct PG/TG matrices",
)

## Paired differences from unrestricted generation

Each comparison uses the same title, PG/TG cell and seed combination. Each
heatmap shows style accuracy minus unrestricted accuracy in percentage points.

In [ ]:
style_effects = {}
for style in ["Photorealistic", "Sketch", "Comic"]:
    style_effects[style] = paired_style_analysis(
        titles[style],
        titles["Unrestricted"],
        style,
        "Unrestricted",
        style.lower(),
        OUTPUT_DIR,
    )
sensitivity_style_effects = {}
for style in ["Photorealistic", "Sketch", "Comic"]:
    sensitivity_style_effects[f"sensitivity_style_effects_{style.lower()}"] = pd.concat([
        paired_style_analysis(
            titles[style], titles["Unrestricted"], style, "Unrestricted",
            style.lower(), OUTPUT_DIR, metric=m, plot=False,
        ).assign(metric=m) for m in SENSITIVITY_METRICS
    ], ignore_index=True)


## Exploratory Qwen PG style comparison

Compare Q38 and Q25 PG changes from Unrestricted for each fixed style, averaging
all four TG models per title. These three exploratory comparisons use pointwise and
approximate familywise 95% Bonferroni-percentile intervals. They concern verified
title recovery, not measured style adherence.


In [ ]:
pg_style_followup = qwen_pg_style_followup(title_scores)
pg_style_followup["exploratory_qwen_pg_style_interactions"]


## Supporting data

Exports record coverage, deduplicated verification counts, errors, timings and provenance.
Centrality describes equal-weight distances from other styles without selecting one.
Keep `style_summary.json`, `manifest.json` and their linked artifacts together for
validation and import into `final_study.ipynb`.


In [ ]:
technical = technical_tables(observations, titles, list(jobs.items()))
export_tables(
    {
        **technical,
        **pg_style_followup,
        **sensitivity_style_effects,
        **{f"style_effects_{name.lower()}": table for name, table in style_effects.items()},
        "style_accuracy": accuracy,
        "style_centrality": centrality,
        "style_title_scores": title_scores,
        "style_prompt_verifications": prompt_verifications(
            pd.concat(observations.values(), ignore_index=True)
        ),
    },
    OUTPUT_DIR,
)
write_manifest(
    ROOT / "notebooks/style_decision.ipynb",
    STYLES,
    OUTPUT_DIR,
    analysis={
        "purpose": "complete_style_comparison",
        "local_image_file_check": "required" if REQUIRE_IMAGE_FILES else "omitted_metadata_only",
        "seed_observations_per_title": 4,
        "model_pairs": [f"{pg}/{bi}" for pg in QG for bi in QG],
        "denominator": "all planned direct observations for every verification and title-match policy",
        "primary_metric": STYLE_PRIMARY_METRIC,
        "study_revision": "final_v12",
        "exploratory_followup": {
            "question": "Q38 minus Q25 PG style-minus-Unrestricted effect",
            "styles": ["Photorealistic", "Sketch", "Comic"],
            "family_size": 3,
            "interval": "Bonferroni percentile approximate familywise 95%",
            "timing": "specified after earlier result review",
        },
        "sensitivity_metrics": list(SENSITIVITY_METRICS),
        "primary_definition": STYLE_PRIMARY_DEFINITION,
        "style_analysis_methods": STYLE_ANALYSIS_METHODS,
        "bootstrap": STYLE_BOOTSTRAP,
    },
)
export_style_summary(OUTPUT_DIR)
